In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

dbutils.widgets.text("catalog", "sentinel_dev")

CATALOG = dbutils.widgets.get("catalog")

SILVER_CURRENT = f"{CATALOG}.silver.silver_orders_current"
SILVER_HISTORY = f"{CATALOG}.silver.silver_orders_history"
VALIDATED = f"{CATALOG}.silver.silver_orders_validated"
QUARANTINE = f"{CATALOG}.silver.orders_quarantine"
GOLD_FACT = f"{CATALOG}.gold.fact_orders"

HEALTH_TABLE = f"{CATALOG}.monitoring.pipeline_health"

print(f"Environment catalog: {CATALOG}")

In [0]:
silver_current_count = spark.table(SILVER_CURRENT).count()
silver_history_count = spark.table(SILVER_HISTORY).count()
validated_count = spark.table(VALIDATED).count()
quarantine_count = spark.table(QUARANTINE).count()
gold_count = spark.table(GOLD_FACT).count()

total_dq_records = validated_count + quarantine_count

quality_pass_rate = (
    validated_count / total_dq_records
    if total_dq_records > 0
    else 1.0
)

print(f"Silver current : {silver_current_count:,}")
print(f"Silver history : {silver_history_count:,}")
print(f"Validated      : {validated_count:,}")
print(f"Quarantined    : {quarantine_count:,}")
print(f"Gold           : {gold_count:,}")
print(f"DQ pass rate   : {quality_pass_rate:.2%}")

In [0]:
freshness = (
    spark.table(SILVER_CURRENT)
        .agg(
            F.max("ingested_at").alias("latest_ingestion"),
            F.max("order_timestamp_clean").alias(
                "latest_business_event"
            )
        )
        .first()
)

latest_ingestion = freshness["latest_ingestion"]
latest_business_event = freshness["latest_business_event"]

print(f"Latest ingestion      : {latest_ingestion}")
print(f"Latest business event : {latest_business_event}")

In [0]:
# %sql
# ALTER TABLE sentinel_dev.monitoring.pipeline_health
# ADD COLUMNS (
#     pipeline_name STRING,
#     gold_count BIGINT,
#     latest_ingestion TIMESTAMP,
#     latest_business_event TIMESTAMP
# );

In [0]:
from datetime import datetime, timezone
from pyspark.sql import Row

health_df = spark.createDataFrame([
    Row(
        # Existing monitoring schema
        current_orders=int(silver_current_count),
        historical_versions=int(silver_history_count),
        validated_records=int(validated_count),
        quarantined_records=int(quarantine_count),
        quality_pass_rate=float(quality_pass_rate),
        metric_timestamp=datetime.now(timezone.utc),

        # Newly added monitoring columns
        pipeline_name="sentinel_orders",
        gold_count=int(gold_count),
        latest_ingestion=latest_ingestion,
        latest_business_event=latest_business_event
    )
])

print("Health DataFrame schema:")
health_df.printSchema()

display(health_df)

In [0]:
(
    health_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "sentinel_dev.monitoring.pipeline_health"
        )
)

print("Pipeline health snapshot recorded successfully.")

In [0]:
print("DATAFRAME:")
health_df.printSchema()

print("TARGET TABLE:")
spark.table(
    "sentinel_dev.monitoring.pipeline_health"
).printSchema()